<a href="https://colab.research.google.com/github/JuanDiegoPasos/proyecto-integrador-pc/blob/main/notebooks/analisis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto final de Introducción a la Programación Científica

## Resultados tipo Saber 11 en Antioquia (2025)

**Estudiante:** Juan Diego Pasos Gómez


## Contexto del proyecto

En este trabajo reviso los resultados de 3.250 estudiantes que presentaron una prueba tipo Saber 11 en distintos municipios de Antioquia durante 2025. La idea es comparar sus puntajes y mirar qué características aparecen relacionadas con esos resultados.

En un caso real, este tipo de información podría servir como punto de partida para identificar grupos de estudiantes que necesitan más apoyo académico o mejores recursos para estudiar.

La base no incluye el nombre de cada colegio. Por eso no puedo señalar instituciones específicas, pero sí comparar los resultados por estrato, zona, tipo de colegio y jornada.

Hay que tener presente que los datos son sintéticos. Sirven para practicar el análisis y sacar conclusiones sobre esta base, pero no representan los resultados reales de los estudiantes de Antioquia.


# 1. Carga e inspección

Primero cargo el archivo CSV y reviso cómo está organizada la información. **shape** muestra cuántas filas y columnas tiene la base. **head()** permite ver los primeros registros, **info()** muestra los tipos de datos y cuántos valores tiene cada columna, y **describe()** resume las variables numéricas.

También reviso cuántos estudiantes hay en algunas categorías. Esto ayuda a entender el tamaño de los grupos antes de comparar sus resultados.

**Antes de ejecutar:** crea una carpeta llamada **data** en los archivos de Colab y sube allí el CSV.


In [ ]:
import pandas as pd
import numpy as np

# Cargar el archivo y revisar su estructura
df_saber = pd.read_csv("data/educacion_saber11_antioquia_2025.csv")

print("Filas y columnas:", df_saber.shape)
print("=" * 80)

print("Primeros registros:")
print(df_saber.head())
print("=" * 80)

print("Tipos de datos:")
print(df_saber.dtypes)
print("=" * 80)

df_saber.info()
print("=" * 80)

print("Resumen de las columnas numéricas:")
print(df_saber.describe().round(1))


Filas y columnas: (3250, 23)
Primeros registros:
  id_estudiante municipio  ... puntaje_ingles  puntaje_global
0     EST-00001  Medellín  ...           51.0             246
1     EST-00002  Medellín  ...           42.0             258
2     EST-00003  Envigado  ...           39.0             246
3     EST-00004  Medellín  ...           51.0             211
4     EST-00005  Medellín  ...           64.0             243

[5 rows x 23 columns]
Tipos de datos:
id_estudiante                   object
municipio                       object
zona                            object
estrato                          int64
tipo_colegio                    object
jornada                         object
sexo                            object
edad                             int64
educacion_madre                 object
ingreso_familiar_smmlv         float64
acceso_internet                 object
computador_hogar                object
trabaja                         object
horas_trabajo_semana           fl

In [ ]:
# Revisar cuántos estudiantes hay en algunos grupos
print("Estudiantes por jornada:")
print(df_saber["jornada"].value_counts())

print("=" * 80)

print("Estudiantes por zona:")
print(df_saber["zona"].value_counts())

print("=" * 80)

print("Estudiantes por tipo de colegio:")
print(df_saber["tipo_colegio"].value_counts())

print("=" * 80)

print("Estudiantes por estrato:")
print(df_saber["estrato"].value_counts().sort_index())

print("=" * 80)

print("Situación laboral:")
print(df_saber["trabaja"].value_counts())


Estudiantes por jornada:
jornada
Mañana      1086
Tarde        731
Única        629
Completa     484
Nocturna     214
Sabatina     106
Name: count, dtype: int64
Estudiantes por zona:
zona
Urbana    2748
Rural      502
Name: count, dtype: int64
Estudiantes por tipo de colegio:
tipo_colegio
Oficial    2601
Privado     649
Name: count, dtype: int64
Estudiantes por estrato:
estrato
1    1285
2    1146
3     577
4     146
5      57
6      39
Name: count, dtype: int64
Situación laboral:
trabaja
No    2495
Sí     755
Name: count, dtype: int64


### Primera revisión

La base tiene **3.250 estudiantes y 23 columnas**. El puntaje global tiene un promedio de **262,8** y una mediana de **262**. Como los dos valores son cercanos, en esta primera revisión no se ve una diferencia grande entre el promedio y el valor que queda en la mitad de los datos.

Los grupos tienen tamaños distintos. Por ejemplo, hay **1.086 estudiantes en jornada de mañana** y solo **106 en sabatina**. Esto importa porque un promedio de un grupo pequeño puede cambiar más fácilmente que el de uno con muchos estudiantes.

Con esta primera mirada ya conozco un poco mejor la base. Antes de comparar resultados, falta revisar si hay datos incompletos o valores que no tengan sentido.


## 2. Calidad de los datos

En esta parte reviso qué información falta y si hay valores que llamen la atención. No todos los espacios vacíos significan lo mismo. A veces falta una respuesta y otras veces el dato simplemente no aplica.

También reviso si hay filas o identificadores repetidos, códigos que puedan representar errores y puntajes fuera de los rangos esperados.


In [ ]:
# Cantidad y porcentaje de datos faltantes por columna
tabla_faltantes = pd.DataFrame({
    "faltantes": df_saber.isna().sum(),
    "porcentaje": (df_saber.isna().mean() * 100).round(2)
})

tabla_faltantes = tabla_faltantes[tabla_faltantes["faltantes"] > 0]
print(tabla_faltantes.sort_values("porcentaje", ascending=False))

print("=" * 80)


print("\nFaltantes en educación de la madre por zona (%):")
print((df_saber["educacion_madre"].isna().groupby(df_saber["zona"]).mean() * 100).round(1))

                        faltantes  porcentaje
horas_trabajo_semana         2495       76.77
ingreso_familiar_smmlv       1845       56.77
educacion_madre               344       10.58
horas_estudio_semana          270        8.31
inasistencias_anio            227        6.98
puntaje_ingles                122        3.75
acceso_internet                92        2.83

Faltantes en educación de la madre por zona (%):
zona
Rural     17.5
Urbana     9.3
Name: educacion_madre, dtype: float64


In [ ]:
# Revisar si los vacíos de horas de trabajo significan "no aplica"
print("Estudiantes según si trabajan:")
print(df_saber["trabaja"].value_counts())

print("=" * 80)


print("\nHoras de trabajo registradas en cada grupo:")
print(df_saber.groupby("trabaja")["horas_trabajo_semana"].count())

print("=" * 80)


print("\nFilas duplicadas:", df_saber.duplicated().sum())
print("IDs duplicados:", df_saber["id_estudiante"].duplicated().sum())

Estudiantes según si trabajan:
trabaja
No    2495
Sí     755
Name: count, dtype: int64

Horas de trabajo registradas en cada grupo:
trabaja
No      0
Sí    755
Name: horas_trabajo_semana, dtype: int64

Filas duplicadas: 0
IDs duplicados: 0


In [ ]:
# Revisar códigos centinela, números negativos y rangos de los puntajes
columnas_numericas = [
    "estrato",
    "edad",
    "ingreso_familiar_smmlv",
    "horas_trabajo_semana",
    "horas_estudio_semana",
    "inasistencias_anio",
    "puntaje_matematicas",
    "puntaje_lectura_critica",
    "puntaje_ciencias_naturales",
    "puntaje_sociales_ciudadanas",
    "puntaje_ingles",
    "puntaje_global"
]

codigos_centinela = [-999, -99, -1, 999, 9999]

print("Códigos centinela revisados:")
print(df_saber[columnas_numericas].isin(codigos_centinela).sum())
print("=" * 80)

print("Valores negativos por columna:")
print((df_saber[columnas_numericas] < 0).sum())
print("=" * 80)

columnas_puntajes = [
    "puntaje_matematicas",
    "puntaje_lectura_critica",
    "puntaje_ciencias_naturales",
    "puntaje_sociales_ciudadanas",
    "puntaje_ingles"
]

print("Mínimos y máximos de los puntajes por área:")
print(df_saber[columnas_puntajes].agg(["min", "max"]))
print("=" * 80)

print("Mínimo y máximo del puntaje global:")
print(df_saber["puntaje_global"].agg(["min", "max"]))


Códigos centinela revisados:
estrato                        0
edad                           0
ingreso_familiar_smmlv         0
horas_trabajo_semana           0
horas_estudio_semana           0
inasistencias_anio             0
puntaje_matematicas            0
puntaje_lectura_critica        0
puntaje_ciencias_naturales     0
puntaje_sociales_ciudadanas    0
puntaje_ingles                 0
puntaje_global                 0
dtype: int64
Valores negativos por columna:
estrato                        0
edad                           0
ingreso_familiar_smmlv         0
horas_trabajo_semana           0
horas_estudio_semana           0
inasistencias_anio             0
puntaje_matematicas            0
puntaje_lectura_critica        0
puntaje_ciencias_naturales     0
puntaje_sociales_ciudadanas    0
puntaje_ingles                 0
puntaje_global                 0
dtype: int64
Mínimos y máximos de los puntajes por área:
     puntaje_matematicas  ...  puntaje_ingles
min                 13.0  ...   

In [ ]:
# Revisar valores extremos en las horas de estudio
print(df_saber["horas_estudio_semana"].describe())

q1 = df_saber["horas_estudio_semana"].quantile(0.25)
q3 = df_saber["horas_estudio_semana"].quantile(0.75)
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr

print("=" * 80)


print("\nLímite superior del IQR:", limite_superior)

print("=" * 80)

print("Valores por encima del límite:", (df_saber["horas_estudio_semana"] > limite_superior).sum())

print("=" * 80)

print("\nRegistros con más de 40 horas de estudio por semana:")

print(df_saber.loc[
    df_saber["horas_estudio_semana"] > 40,
    ["edad", "jornada", "trabaja", "horas_estudio_semana"]
])

print("=" * 80)

# Revisar a los estudiantes adultos
adultos = df_saber.loc[df_saber["edad"] >= 22]
print("\nEstudiantes de 22 años o más:", len(adultos))
print(adultos["jornada"].value_counts())

count    2980.000000
mean        7.186242
std         6.456445
min         0.000000
25%         4.000000
50%         6.000000
75%         9.000000
max       168.000000
Name: horas_estudio_semana, dtype: float64

Límite superior del IQR: 16.5
Valores por encima del límite: 105

Registros con más de 40 horas de estudio por semana:
      edad jornada trabaja  horas_estudio_semana
327     16  Mañana      Sí                 168.0
986     18  Mañana      No                 168.0
2984    16   Tarde      No                  90.0
3016    15   Tarde      No                  90.0

Estudiantes de 22 años o más: 264
jornada
Nocturna    175
Sabatina     89
Name: count, dtype: int64


### Problemas encontrados

Hay **siete columnas con datos faltantes**. La que más preocupa es **ingreso_familiar_smmlv**, porque le falta el **56,77 %** de la información. En cambio, los **2.495** espacios vacíos de **horas_trabajo_semana** coinciden con los estudiantes que dijeron que no trabajan. En esos casos las horas no aplican, así que completarlas sería inventar un dato.

En **horas_estudio_semana** faltan 270 valores. También aparecen cuatro registros de **90 y 168 horas por semana**. El IQR señala 105 valores que se alejan del resto, pero eso no significa que todos estén mal. Al revisar los casos más extremos, esos cuatro valores por encima de 40 horas son los que voy a tratar como sospechosos. El siguiente valor más alto después de ellos es de 29 horas.

En **educacion_madre** faltan 344 respuestas. Estos espacios aparecen con más frecuencia en la zona rural (**17,5 %**) que en la urbana (**9,3 %**). También hay 227 datos faltantes en **inasistencias_anio**, 122 en **puntaje_ingles** y 92 en **acceso_internet**.

No encontré filas duplicadas ni IDs repetidos. Tampoco aparecieron los códigos centinela que revisé ni valores negativos en las columnas numéricas. Los puntajes están dentro de sus rangos: de 0 a 100 por área y de 0 a 500 en el puntaje global.

También hay **264 estudiantes de 22 años o más**. Todos están en jornada nocturna o sabatina, así que los conservo. Por lo que muestra la base, parecen corresponder a estudiantes adultos y no a errores de edad.


## 3. Limpieza

Para hacer los cambios trabajo sobre una copia de la base original. Así puedo conservar los datos como llegaron y revisar qué cambió después de la limpieza.

- **Ingreso familiar:** elimino esta columna porque más de la mitad de sus datos están vacíos. El estrato permite hacer algunas comparaciones socioeconómicas, aunque no reemplaza el dato exacto del ingreso. Para las preguntas de este trabajo tampoco necesito esa columna.

- **Horas de estudio:** primero marco como faltantes los cuatro registros de 90 y 168 horas, porque son valores muy poco creíbles para una semana. Después completo los datos faltantes con la mediana según si el estudiante trabaja o no. Uso esa separación porque los dos grupos tienen medianas distintas: 7 horas para quienes no trabajan y 4 para quienes sí.

- **Educación de la madre:** completo los faltantes con la categoría que más se repite dentro de cada estrato. Como es una variable de categorías, no tendría sentido sacar un promedio. Además, la categoría más común cambia entre estratos.

No completo `horas_trabajo_semana`, porque sus espacios vacíos significan que el estudiante no trabaja. Tampoco completo los datos de internet, inasistencias ni inglés. Es mejor conservar esos faltantes que inventar respuestas que no tenemos.


In [ ]:
# Crear una copia y guardar el tamaño inicial
df_limpio = df_saber.copy()
filas_antes, columnas_antes = df_limpio.shape

# Eliminar la columna con más datos faltantes
df_limpio = df_limpio.drop(columns=["ingreso_familiar_smmlv"])

# Marcar como faltantes las cuatro horas de estudio sospechosas
horas_sospechosas = df_limpio["horas_estudio_semana"] > 40
cantidad_sospechosas = int(horas_sospechosas.sum())
df_limpio.loc[horas_sospechosas, "horas_estudio_semana"] = np.nan

# Completar horas de estudio con la mediana según si trabaja
imputados_estudio = int(df_limpio["horas_estudio_semana"].isna().sum())
mediana_por_trabajo = df_limpio.groupby("trabaja")["horas_estudio_semana"].transform("median")
df_limpio["horas_estudio_semana"] = df_limpio["horas_estudio_semana"].fillna(mediana_por_trabajo)

# Completar educación de la madre con la moda de cada estrato
imputados_educacion = int(df_limpio["educacion_madre"].isna().sum())

for estrato in range(1, 7):
    moda_estrato = df_limpio.loc[
        df_limpio["estrato"] == estrato, "educacion_madre"
    ].mode()[0]

    faltantes_estrato = (df_limpio["estrato"] == estrato) & df_limpio["educacion_madre"].isna()
    df_limpio.loc[faltantes_estrato, "educacion_madre"] = moda_estrato

    print("Estrato", estrato, "- moda utilizada:", moda_estrato)

print("=" * 80)

print("Mediana de horas de estudio por situación laboral:")
print(df_limpio.groupby("trabaja")["horas_estudio_semana"].median())
print("=" * 80)

print("Filas antes y después:", filas_antes, "->", len(df_limpio))
print("Columnas antes y después:", columnas_antes, "->", df_limpio.shape[1])
print("Horas de estudio completadas:", imputados_estudio)
print("Datos de educación de la madre completados:", imputados_educacion)
print("Total de valores completados:", imputados_estudio + imputados_educacion)
print("=" * 80)

print("Faltantes que se conservan:")
print(df_limpio.isna().sum()[df_limpio.isna().sum() > 0])


Estrato 1 - moda utilizada: Bachillerato
Estrato 2 - moda utilizada: Bachillerato
Estrato 3 - moda utilizada: Técnica o tecnológica
Estrato 4 - moda utilizada: Profesional
Estrato 5 - moda utilizada: Posgrado
Estrato 6 - moda utilizada: Posgrado
Mediana de horas de estudio por situación laboral:
trabaja
No    7.0
Sí    4.0
Name: horas_estudio_semana, dtype: float64
Filas antes y después: 3250 -> 3250
Columnas antes y después: 23 -> 22
Horas de estudio completadas: 274
Datos de educación de la madre completados: 344
Total de valores completados: 618
Faltantes que se conservan:
acceso_internet           92
horas_trabajo_semana    2495
inasistencias_anio       227
puntaje_ingles           122
dtype: int64


### ¿Qué cambió con la limpieza?

La base queda con **3.250 filas y 22 columnas**. No se eliminó ningún estudiante. Solo se quitó la columna de ingresos familiares, que tenía demasiados datos vacíos.

En horas de estudio se completaron **274 valores**. De esos, 270 ya estaban vacíos y cuatro se marcaron como faltantes porque eran valores muy extremos. La mediana utilizada fue de **7 horas** para quienes no trabajan y **4 horas** para quienes sí trabajan.

En educación de la madre se completaron **344 valores** con la moda de cada estrato. En total quedaron completadas **618 celdas** entre las dos columnas.

Todavía hay datos vacíos en internet, horas de trabajo, inasistencias e inglés. Se conservan porque algunos significan que la información no aplica y los demás no necesitan rellenarse para responder las cinco preguntas.

Las dos columnas que se completaron no se usan para calcular los resultados de las cinco preguntas. La limpieza se hace para cumplir el ejercicio y dejar esas variables listas para otros análisis, sin cambiar los resultados que se presentan más adelante.


## 4. Preguntas de análisis

En esta parte uso la base limpia para responder las cinco preguntas del proyecto. Cuando comparo promedios, también muestro **count**, que indica cuántos estudiantes hay en cada grupo. Así se puede ver si un resultado viene de un grupo grande o de pocos registros.


### Pregunta 1 — ¿Cómo cambia el puntaje global según el estrato, la zona y el tipo de colegio?

Primero comparo el puntaje global por estrato, zona y tipo de colegio. Después cruzo la zona con el tipo de colegio para revisar si las diferencias también aparecen al mirar esas dos características juntas. La idea es ver cómo se distribuyen los resultados entre los grupos, no decir que una sola característica explique el puntaje.


In [ ]:
print("Puntaje global por estrato:")
print(df_limpio.groupby("estrato")["puntaje_global"].agg(["count", "mean"]).round(1))

print("=" * 80)

print("\nPuntaje global por zona:")
print(df_limpio.groupby("zona")["puntaje_global"].agg(["count", "mean"]).round(1))

print("=" * 80)

print("\nPuntaje global por tipo de colegio:")
print(df_limpio.groupby("tipo_colegio")["puntaje_global"].agg(["count", "mean"]).round(1))

print("=" * 80)

print("\nPuntaje global por zona y tipo de colegio:")
print(df_limpio.groupby(["zona", "tipo_colegio"])["puntaje_global"].agg(["count", "mean"]).round(1))

Puntaje global por estrato:
         count   mean
estrato              
1         1285  232.0
2         1146  264.5
3          577  295.3
4          146  322.4
5           57  351.5
6           39  393.1

Puntaje global por zona:
        count   mean
zona                
Rural     502  227.7
Urbana   2748  269.2

Puntaje global por tipo de colegio:
              count   mean
tipo_colegio              
Oficial        2601  251.9
Privado         649  306.3

Puntaje global por zona y tipo de colegio:
                     count   mean
zona   tipo_colegio              
Rural  Oficial         415  220.2
       Privado          87  263.6
Urbana Oficial        2186  258.0
       Privado         562  313.0


**¿Qué muestran los resultados?**

La diferencia por estrato es bastante marcada. El promedio pasa de **232,0 puntos en el estrato 1** a **393,1 en el estrato 6**. También hay una diferencia entre zonas: los estudiantes de zona urbana promedian **269,2**, mientras que los de zona rural promedian **227,7**. Por tipo de colegio, el promedio es de **306,3 en los privados** y **251,9 en los oficiales**.

Al cruzar zona y tipo de colegio, el promedio más bajo aparece en los colegios oficiales rurales, con **220,2 puntos** entre 415 estudiantes. En los colegios privados también hay diferencia entre zona rural y urbana, así que no se trata de una sola comparación.

Estos resultados ayudan a ubicar grupos donde valdría la pena revisar las necesidades de apoyo. Eso sí, los estratos 5 y 6 tienen pocos estudiantes, 57 y 39 respectivamente. Por eso sus promedios se deben mirar con cuidado. Además, esta base no permite separar por completo la influencia del estrato, la zona y el tipo de colegio.


### Pregunta 2 — ¿Tener internet o computador en casa se asocia con mejores resultados?

Comparo el puntaje global de quienes tienen internet en casa con el de quienes no tienen. Hago la misma comparación con el computador. Los 92 estudiantes que no respondieron sobre el internet quedan por fuera de ese cálculo, porque no hay forma de saber a cuál grupo pertenecen.


In [ ]:
print("Puntaje según acceso a internet:")
print(df_limpio.groupby("acceso_internet")["puntaje_global"].agg(["count", "mean"]).round(1))

print("=" * 80)

print("\nPuntaje según computador en casa:")
print(df_limpio.groupby("computador_hogar")["puntaje_global"].agg(["count", "mean"]).round(1))

print("=" * 80)

print("\nEstudiantes sin respuesta sobre internet:", df_limpio["acceso_internet"].isna().sum())

Puntaje según acceso a internet:


                 count   mean
acceso_internet              
No                1578  244.6
Sí                1580  280.8

Puntaje según computador en casa:
                  count   mean
computador_hogar              
No                 2060  251.2
Sí                 1190  282.9

Estudiantes sin respuesta sobre internet: 92


**¿Qué muestran los resultados?**

Aquí aparece una diferencia clara. Quienes tienen internet promedian **280,8 puntos**, mientras que quienes no tienen promedian **244,6**. La diferencia es de **36,2 puntos**. Con el computador pasa algo parecido: quienes tienen uno promedian **282,9** y quienes no tienen promedian **251,2**, una diferencia de **31,7 puntos**.

En esta base, tener internet o computador aparece relacionado con mejores resultados. Por eso, el acceso a estos recursos puede ser una pista para identificar estudiantes que podrían necesitar apoyo. No significa que entregar internet vaya a subir automáticamente el puntaje, porque también pueden influir otras condiciones del hogar y del colegio.


### Pregunta 3 — ¿Los estudiantes que trabajan obtienen puntajes más bajos? ¿Depende de cuántas horas trabajan?

Primero comparo el promedio de quienes trabajan con el de quienes no trabajan. Después reviso solo a los estudiantes que sí trabajan para mirar si las horas de trabajo se relacionan con el puntaje global.

Este filtro es importante porque **horas_trabajo_semana** está vacía para quienes no trabajan. En esos casos el dato no aplica. Por eso no mezclo esos espacios vacíos con las horas de quienes sí tienen un trabajo registrado.


In [ ]:
print("Puntaje según situación laboral:")
print(df_limpio.groupby("trabaja")["puntaje_global"].agg(["count", "mean", "median"]).round(1))

print("=" * 80)

# Quedarnos solo con estudiantes que trabajan
trabajadores = df_limpio.loc[df_limpio["trabaja"] == "Sí"]

print("\nEstudiantes que trabajan:", len(trabajadores))
print("Correlación entre horas trabajadas y puntaje global:")
print(trabajadores[["horas_trabajo_semana", "puntaje_global"]].corr().round(2))

Puntaje según situación laboral:
         count   mean  median
trabaja                      
No        2495  272.0   268.0
Sí         755  232.3   230.0

Estudiantes que trabajan: 755
Correlación entre horas trabajadas y puntaje global:
                      horas_trabajo_semana  puntaje_global
horas_trabajo_semana                  1.00           -0.15
puntaje_global                       -0.15            1.00


**¿Qué muestran los resultados?**

Los estudiantes que no trabajan obtienen **272,0 puntos en promedio** y son 2.495. Quienes sí trabajan promedian **232,3 puntos** y son 755. La diferencia entre los dos grupos es de **39,7 puntos**.

Entre quienes trabajan, la correlación entre las horas trabajadas y el puntaje es **-0,15**. Es una relación negativa débil. En general, el puntaje tiende a bajar un poco cuando aumentan las horas de trabajo, pero la relación no es fuerte.

Esto muestra que vale la pena revisar las necesidades de quienes combinan estudio y trabajo. De todas formas, no puedo afirmar que trabajar sea la causa de los puntajes más bajos. También pueden influir la edad y las condiciones familiares de estos estudiantes.


### Pregunta 4 — ¿Qué tan relacionados están los puntajes de las distintas áreas entre sí?

Calculo la correlación entre matemáticas, lectura crítica, ciencias naturales, sociales e inglés. Este valor va de -1 a 1. Cuando es positivo, los dos puntajes suelen subir juntos. Cuando es negativo, uno tiende a bajar mientras el otro sube. Si está cerca de cero, la relación lineal es baja.

No completo los 122 puntajes de inglés que faltan. Para cada pareja de áreas, pandas calcula la correlación usando los estudiantes que sí tienen los dos puntajes. Así no tengo que inventar resultados.


In [ ]:
print("Matriz de correlaciones entre las cinco áreas:")
print(df_limpio[columnas_puntajes].corr().round(2))

print("=" * 80)

print("\nPuntajes de inglés faltantes:", df_limpio["puntaje_ingles"].isna().sum())

Matriz de correlaciones entre las cinco áreas:


                             puntaje_matematicas  ...  puntaje_ingles
puntaje_matematicas                         1.00  ...            0.69
puntaje_lectura_critica                     0.68  ...            0.67
puntaje_ciencias_naturales                  0.69  ...            0.68
puntaje_sociales_ciudadanas                 0.69  ...            0.69
puntaje_ingles                              0.69  ...            1.00

[5 rows x 5 columns]

Puntajes de inglés faltantes: 122


**¿Qué muestran los resultados?**

Las correlaciones entre las cinco áreas están entre **0,66 y 0,69**. Todas son positivas y bastante parecidas. Esto quiere decir que, en esta base, los estudiantes que tienen buenos resultados en un área suelen tener también buenos resultados en las demás.

Por eso, las diferencias de desempeño no parecen concentrarse en una sola materia. Podría tener sentido pensar en apoyos académicos generales, sin dejar de revisar las necesidades de cada área.

Los 122 puntajes faltantes de inglés no entran en los cálculos que incluyen esa materia. Por eso esas correlaciones se calculan con 3.128 estudiantes. Las relaciones entre las otras cuatro áreas usan los 3.250 registros.


### Pregunta 5 — ¿Qué jornadas necesitan más apoyo? ¿Quiénes estudian en ellas?

Ordeno las jornadas según el promedio del puntaje global para encontrar cuáles tienen los resultados más bajos. Después reviso la edad y cuántos estudiantes trabajan en cada jornada. Así puedo mirar no solo cómo les va, sino también algunas características de quienes estudian allí.


In [ ]:
print("Puntaje global por jornada, de menor a mayor:")
print(
    df_limpio.groupby("jornada")["puntaje_global"]
    .agg(["count", "mean", "median"])
    .sort_values("mean")
    .round(1)
)

print("=" * 80)

print("Edad por jornada:")
print(df_limpio.groupby("jornada")["edad"].agg(["mean", "median"]).round(1))
print("=" * 80)

print("Cantidad de estudiantes que trabajan y no trabajan por jornada:")
print(df_limpio.groupby(["jornada", "trabaja"])["id_estudiante"].count())

# Comparar nocturna y sabatina con las demás jornadas
jornadas_prioritarias = (
    (df_limpio["jornada"] == "Nocturna")
    | (df_limpio["jornada"] == "Sabatina")
)

puntaje_nocturna_sabatina = df_limpio.loc[jornadas_prioritarias, "puntaje_global"]
puntaje_otras_jornadas = df_limpio.loc[~jornadas_prioritarias, "puntaje_global"]

print("=" * 80)

print("Nocturna y sabatina:")
print("Estudiantes:", len(puntaje_nocturna_sabatina), "| Promedio:", round(puntaje_nocturna_sabatina.mean(), 1))
print("Otras jornadas:")
print("Estudiantes:", len(puntaje_otras_jornadas), "| Promedio:", round(puntaje_otras_jornadas.mean(), 1))


Puntaje global por jornada, de menor a mayor:
          count   mean  median
jornada                       
Sabatina    106  225.1   229.0
Nocturna    214  230.7   228.5
Tarde       731  256.4   259.0
Única       629  256.8   257.0
Mañana     1086  261.4   260.0
Completa    484  305.9   305.0
Edad por jornada:
          mean  median
jornada               
Completa  16.8    17.0
Mañana    16.8    17.0
Nocturna  29.7    29.0
Sabatina  30.6    30.0
Tarde     16.8    17.0
Única     16.9    17.0
Cantidad de estudiantes que trabajan y no trabajan por jornada:
jornada   trabaja
Completa  No         409
          Sí          75
Mañana    No         875
          Sí         211
Nocturna  No          53
          Sí         161
Sabatina  No          32
          Sí          74
Tarde     No         609
          Sí         122
Única     No         517
          Sí         112
Name: id_estudiante, dtype: int64
Nocturna y sabatina:
Estudiantes: 320 | Promedio: 228.9
Otras jornadas:
Estudiantes: 293

**¿Qué muestran los resultados?**

Los promedios más bajos aparecen en las jornadas **Sabatina (225,1 puntos, 106 estudiantes)** y **Nocturna (230,7 puntos, 214 estudiantes)**. Entre las dos reúnen 320 estudiantes y tienen un promedio de **228,9 puntos**. Las demás jornadas promedian **266,5**.

También se ve que son grupos de estudiantes adultos. La edad promedio es de **29,7 años en nocturna** y **30,6 en sabatina**. En la jornada nocturna trabajan 161 de 214 estudiantes, es decir, el **75,2 %**. En sabatina trabajan 74 de 106, el **69,8 %**. Además, todos los estudiantes de estas dos jornadas pertenecen a colegios oficiales.

Esto ayuda a entender mejor las necesidades de estos grupos. Podrían servir los apoyos académicos en horarios flexibles y que tengan en cuenta que muchos estudiantes también trabajan. No puedo decir que estudiar de noche o los sábados sea la causa del menor promedio, porque la edad y la situación laboral también son diferentes frente a las jornadas diurnas.


## 5. Conclusiones parciales

En esta base, las diferencias más marcadas aparecen entre los estratos, las zonas y los tipos de colegio. También hay diferencias entre estudiantes que tienen internet o computador en casa y quienes no cuentan con esos recursos. Algo parecido ocurre al comparar a quienes trabajan con quienes no.

Las jornadas nocturna y sabatina son las que tienen los promedios más bajos. Además, reúnen estudiantes de mayor edad y una proporción alta de personas que trabajan. Por eso, serían dos grupos importantes para revisar al pensar en tutorías o apoyos que se adapten a sus horarios. La conectividad también aparece como un tema que vale la pena tener en cuenta.

Este análisis sirve como punto de partida para las gráficas de la segunda entrega. Los resultados no prueban que una característica sea la causa de otra y tampoco describen la situación real de Antioquia, porque los datos son sintéticos. También hay grupos pequeños, como los estratos 5 y 6 y la jornada sabatina, cuyos promedios se deben interpretar con cuidado.
